In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pypdf import PdfReader

# Load French-capable LLM
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def read_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

def ask_llm_from_pdf(pdf_path, question):
    document = read_pdf(pdf_path)

    prompt = f"""
Voici un document en français :

{document}

Question :
{question}

Réponds uniquement en te basant sur le document.
Réponds en français.
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(
        **inputs,
        max_length=300
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage
pdf_file = "Chapitre 4.pdf"   # <-- your PDF file
question = "Résume le contenu du document."

answer = ask_llm_from_pdf(pdf_file, question)
print(answer)


A document in French: 1 Chapitre 4 : Generality on the 3D Printing 1. Introduction The 3D printing technology, also called additive fabrication, is a technology of fabrication which allows to create physical objects based on a model numérique. Contrary to traditional methods of fabrication soustractive (usinage, découpe), the 3D printing process built the object couched by couche, en adding the material only where it is necessary. This procédé uses different materials according to the technology employed: plastics (PLA, ABS), resins, metals, composites, céramiques, or even biological materials. Since its emergence in the years 1980, l’impression 3D is devenue ie accessible, polyvalent and largely repandue in industry, education and health. 2. Utilisations of 3D Printing 3D is used in many areas owing to its ability to create quickly customized and complex objects. In architecture and construction, it provides for precise maquettes, models anatomiques and for the development of bio-impr

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from pypdf import PdfReader

# Charger le modèle multilingue MBART
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

# Forcing French generation
tokenizer.src_lang = "fr_XX"

# Fonction pour lire le PDF
def read_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

# Fonction pour poser une question et obtenir la réponse
def ask_pdf_french(pdf_text, question):
    # Créer le prompt
    prompt = f"""
Document en français :
{pdf_text}

Question :
{question}

Réponds uniquement en français.
"""
    # Tokenization
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)

    # Génération
    generated_tokens = model.generate(
        **inputs,
        max_length=300,
        forced_bos_token_id=tokenizer.lang_code_to_id["fr_XX"]
    )

    # Décodage
    answer = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    return answer

# ===== Programme interactif =====
pdf_path = input("Entrez le chemin du fichier PDF : ")
pdf_text = read_pdf(pdf_path)

print("\nPDF chargé avec succès.")
print("Tapez votre question (ou 'exit' pour quitter)\n")

while True:
    question = input("Votre question : ")
    if question.lower() == "exit":
        print("Fin du programme.")
        break
    answer = ask_pdf_french(pdf_text, question)
    print("\nRéponse :")
    print(answer)
    print("-" * 50)


Entrez le chemin du fichier PDF : Chapitre 4.pdf

PDF chargé avec succès.
Tapez votre question (ou 'exit' pour quitter)

Votre question : role de l'extredeur

Réponse :
En français : 1 Chapitre 4 : Généralité sur l’impression 3D 1. Introduction 3D, also called additive fabrication, est une technologie de fabrication qui permet de créer des objets physiques à partir d’un modèle numérique. Contrairement aux méthodes traditionnelles de fabrication soustractive (usinage, découpe), l’impression 3D construit l’objet couche par couche, en ajoutant la matière uniquement là où elle est nécessaire. Ce procédé utilise différents matériaux selon la technologie employée : plastics (PLA, ABS), résines, métaux, composites, céramiques, ou même des matériaux biologiques. Depuis son apparition dans les années 1980, l’impression 3D est devenu une technologie accessible, polyvalente et largement répandue dans l’industrie, l’enseignement et la santé. 2. Utilisations de l’impression 3D L’impression 3D est u

In [26]:
# =========================
# Imports
# =========================
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import re

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 1️⃣ Strong text cleaning
# =========================
def clean_text(text):
    text = re.sub(r'\[[0-9]+\]', '', text)  # Remove references [12]
    text = re.sub(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,})\b', '', text)  # Remove titles
    text = re.sub(r'\b(i|ii|iii|iv|v|vi|vii|viii|ix|x)\.\b', '', text, flags=re.I)  # Remove roman numerals
    text = re.sub(r'[•▪–—]', '', text)  # Remove bullets
    text = re.sub(r'(includes|such as|following|consists of)\s*:?','', text, flags=re.I)  # Remove list introducers
    text = re.sub(r'\s+', ' ', text)  # Normalize whitespace
    return text.strip()

# =========================
# 1️⃣a Clean chunks further
# =========================
def clean_chunk(chunk):
    chunk = re.sub(r'^\d+\s+.*$', '', chunk, flags=re.MULTILINE)  # Remove numbered lines
    chunk = re.sub(r'\n+', ' ', chunk)  # Remove extra line breaks
    chunk = re.sub(r'\s+', ' ', chunk)  # Remove multiple spaces
    return chunk.strip()

# =========================
# 2️⃣ Read PDF
# =========================
def read_pdf(path):
    reader = PdfReader(path)
    full_text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + " "
    return clean_text(full_text)

# =========================
# 3️⃣ Chunking (long PDFs)
# =========================
def chunk_text(text, max_words=1000, overlap=250):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words - overlap):
        chunk = " ".join(words[i:i + max_words])
        if len(chunk.split()) > 60:
            chunks.append(clean_chunk(chunk))
    return chunks

# =========================
# 4️⃣ Embedding model
# =========================
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

# =========================
# 5️⃣ LLM model
# =========================
MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# =========================
# 6️⃣ Post-process answer
# =========================
def post_process_answer(answer, question):
    if 'list' in question.lower() or 'classification' in question.lower():
        answer = re.sub(r'[“”"]', '', answer)
    else:
        answer = re.sub(r'\d+/\d+', '', answer)
        answer = re.sub(r'\b\d+\b', '', answer)
        answer = re.sub(r'\b(Note|Then|ELSE|ENDIF|IF|Read|Print|Yes|No)\b', '', answer, flags=re.I)
    answer = re.sub(r'\s+', ' ', answer).strip()

    if len(answer.split()) < 12:
        return "The document mentions this concept, but no clear explanation could be extracted."

    sentences = re.split(r'(?<=[.!?])\s+', answer)
    result = " ".join(sentences[:5])  # Keep first 5 sentences for longer answers

    if result:
        result = result[0].upper() + result[1:]

    return result

# =========================
# 7️⃣ Answer question (RAG)
# =========================
def answer_question(question, chunks, embeddings, top_k=15):
    q_emb = embedder.encode(question, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, embeddings)[0]

    # Retrieve top-k most relevant chunks
    best_scores, best_ids = torch.topk(scores, k=min(top_k, len(chunks)))
    sorted_ids = best_ids[torch.argsort(best_scores, descending=True)]
    candidate_chunks = [chunks[i] for i in sorted_ids]

    # Relaxed keyword filtering for long PDFs
    keywords = re.findall(r'\w+', question.lower())
    if len(keywords) < 3:
        filtered_chunks = [c for c in candidate_chunks if any(k in c.lower() for k in keywords)]
    else:
        filtered_chunks = [c for c in candidate_chunks if any(k in c.lower() for k in keywords)]

    context = " ".join(filtered_chunks[:top_k]) if filtered_chunks else " ".join(candidate_chunks)
    context = re.sub(r'\s+', ' ', context).strip()

    if re.search(r'\b(difference|compare|vs)\b', question, flags=re.I):
        instruction = (
            "Compare the items mentioned in the question. "
            "Explain the differences clearly in full sentences. "
            "Do not list items or quote the text."
        )
    else:
        instruction = (
            "Define the concept asked in the question. "
            "Write a short explanation in full sentences. "
            "Do not list items or quote the text."
        )

    prompt = f"{instruction}\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=600,  # Longer answers for long PDFs
        num_beams=6,
        temperature=0.25,
        repetition_penalty=1.35
    )

    answer = tokenizer.decode(output[0], skip_special_tokens=True)
    return post_process_answer(answer, question)

# =========================
# 8️⃣ PDF summary (section-wise for long PDFs)
# =========================
def summarize_pdf(text):
    summaries = []
    for i in range(0, len(text), 4000):
        section = text[i:i+4000]
        prompt = (
            "Summarize the following document section in English.\n"
            "Use your own words.\n"
            "Limit the summary to 5–7 lines.\n\n"
            f"Text:\n{section}\n\nSummary:"
        )
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            num_beams=5,
            temperature=0.3
        )
        summaries.append(clean_text(tokenizer.decode(output[0], skip_special_tokens=True)))
    return " ".join(summaries)

# =========================
# 9️⃣ Main program
# =========================
if __name__ == "__main__":
    pdf_path = input("Enter PDF path: ")
    pdf_text = read_pdf(pdf_path)
    chunks = chunk_text(pdf_text)
    embeddings = embedder.encode(chunks, convert_to_tensor=True)

    print("\nPDF loaded and indexed. Ask a question ('exit' or 'summary'):\n")
    while True:
        question = input("Your question: ").strip()
        if question.lower() == "exit":
            print("Program finished.")
            break
        elif question.lower() == "summary":
            print("\nSummary:\n")
            print(summarize_pdf(pdf_text))
            print("-" * 70)
        else:
            print("\nAnswer:\n")
            print(answer_question(question, chunks, embeddings))
            print("-" * 70)


Enter PDF path: Chapter3.pdf

PDF loaded and indexed. Ask a question ('exit' or 'summary'):

Your question: what's the static testing

Answer:

Static testing enables the early detection of defects before dynamic testing is performed Defects found early are often much cheaper to remove than defects found later in the lifecycleDevelopment productivity is likely to increasebecause of the less rework effort. Identifying defects which are not easily found by dynamic testingPreventing defects in design or coding by uncovering inconsistencies, ambiguities, contradictions, omissions, inaccuracies, and redundancies in requirementsIncreasing development productivity (e.g., due to improved design, more maintainable code) Benefits of ()7Static testing enables the early detection of defects before dynamic testing is performed Defects found early are often much cheaper to remove than defects found later in the lifecycleDevelopment productivity is likely to increasebecause of the less rework effort.